## Block 5 — The RAG Magic! 🤖
Question + Relevant Chunks → Claude → Answer!

### Load Everything I Built

In [4]:
# Cell 1: Load all components we built in Blocks 1-4
# This brings everything together for RAG

import numpy as np
import json
import faiss
from sentence_transformers import SentenceTransformer

# Load chunks
with open("../data/processed/chunks.json", "r") as f:
    chunks = json.load(f)

# Load FAISS index
index = faiss.read_index("../vector_store/faiss_index/alzheimer.index")

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Chunks loaded:", len(chunks))
print("✅ FAISS index loaded:", index.ntotal, "vectors")
print("✅ Embedding model ready")
print("\nAll components loaded! Ready for RAG!")

c:\Users\tpriy\DatasciencePortfolio\alzheimers-early-detection-rag-chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 853.25it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Chunks loaded: 10
✅ FAISS index loaded: 10 vectors
✅ Embedding model ready

All components loaded! Ready for RAG!


### The Retrieval Function

In [5]:
# Cell 2: Function to retrieve relevant chunks
# Given a question → find most relevant research

def retrieve_chunks(question, top_k=3):
    
    # Step 1: Convert question to vector
    question_vector = model.encode([question])
    question_vector = question_vector.astype(np.float32)
    
    # Step 2: Search FAISS for similar chunks
    distances, indices = index.search(question_vector, k=top_k)
    
    # Step 3: Collect relevant chunks
    relevant_chunks = []
    for dist, idx in zip(distances[0], indices[0]):
        relevant_chunks.append({
            "text": chunks[idx]["text"],
            "title": chunks[idx]["title"],
            "pmid": chunks[idx]["pmid"],
            "distance": float(dist)
        })
    
    return relevant_chunks

# Test it!
results = retrieve_chunks("What are early biomarkers for Alzheimer's?")

print(f"Found {len(results)} relevant chunks:\n")
for i, r in enumerate(results):
    print(f"Chunk {i+1}: {r['title'][:60]}")
    print(f"Distance: {r['distance']:.4f}\n")

Found 3 relevant chunks:

Chunk 1: Tears as a window to Alzheimer's disease: A systematic revie
Distance: 0.8181

Chunk 2: Plasma p-tau217, p-tau181 and Aβ42/40 for Alzheimer's diseas
Distance: 0.8299

Chunk 3: Serum Expression of miR-106b-3p and Its Diagnostic Significa
Distance: 0.8918



### Connect Claude!

In [6]:
# Cell 3: Connect to Claude API
# This is the GENERATION part of RAG

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize Claude client
client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

print("✅ Claude client initialized!")
print("✅ Ready to generate answers!")

✅ Claude client initialized!
✅ Ready to generate answers!


###  Claude client initialized!
 Ready to generate answers!

#### Ask Your First RAG Question!

In [7]:
def ask_alzheimer_chatbot(question):
    
    # Step 1: Retrieve relevant chunks
    relevant_chunks = retrieve_chunks(question, top_k=3)
    
    # Step 2: Build context from chunks
    context = ""
    for i, chunk in enumerate(relevant_chunks):
        context += f"Source {i+1}: {chunk['title']}\n"
        context += f"{chunk['text']}\n\n"
    
    # Step 3: Build prompt for Claude
    prompt = f"""You are an Alzheimer's disease research assistant.
Answer the question using ONLY the provided research context.
Always cite which source you used.
If the context doesn't contain the answer, say so honestly.

RESEARCH CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
    
    # Step 4: Send to Claude
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    return {
        "question": question,
        "answer": response.content[0].text,
        "sources": [r["title"] for r in relevant_chunks]
    }

print("✅ RAG pipeline function ready!")

✅ RAG pipeline function ready!


In [8]:
# Cell 5: Ask your first real RAG question!

result = ask_alzheimer_chatbot(
    "What are early biomarkers for Alzheimer's disease?"
)

print("QUESTION:")
print(result["question"])
print("\nANSWER:")
print(result["answer"])
print("\nSOURCES USED:")
for source in result["sources"]:
    print(f"  → {source}")

QUESTION:
What are early biomarkers for Alzheimer's disease?

ANSWER:
Based on the provided research context, several early biomarkers for Alzheimer's disease have been identified:

**Plasma Biomarkers (Source 1):**
- **Amyloid-beta 42 (Aβ42)** and the **Aβ42/Aβ40 ratio** - these were significantly lower in AD patients and showed positive correlations with cognitive scores
- **Phosphorylated tau 181 (p-tau181)** and **p-tau217** - these were significantly elevated in AD patients and demonstrated negative correlations with cognitive function
- **Glial fibrillary acidic protein (GFAP)** and **neurofilament light chain (NfL)** - also elevated in AD patients

Source 1 notes that "a reduced plasma Aβ42/Aβ40 ratio, along with increased p-tau181 and p-tau217 levels, is significantly associated with a clinical diagnosis of AD, cognitive decline... and positive amyloid deposition on PET-CT."

**Tear Biomarkers (Source 2):**
- **Proteins** found in tears that reflect neurodegenerative processes
